In [8]:
import os
import re
import time
import numpy as np
import pandas as pd
import requests

# =========================
# CONFIG (TEST MODE)
# =========================
FILE_PATH = r"C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\PHARMACIES_COMBINED.csv"

# Option 1: paste key here
OPENCAGE_API_KEY = "937e5ebc14f64695a2d4fffeb36a0448"  # <-- paste your key here, e.g. "123abc..."

# Option 2: or use env var (kernel must be restarted after setx)
if not OPENCAGE_API_KEY:
    OPENCAGE_API_KEY = os.getenv("OPENCAGE_API_KEY", "")

OPENCAGE_URL = "https://api.opencagedata.com/geocode/v1/json"

SLEEP_SECONDS = 1.0
N_TEST = 100
CONFIDENCE_REVIEW_THRESHOLD = 6  # 0-10 typical

PROVINCE_COL = "PROVINCE"


# =========================
# HELPERS
# =========================
def clean_address(addr):
    if pd.isna(addr):
        return ""
    addr = str(addr)

    addr = re.sub(r"\bP\.?\s*O\.?\s*Box\b.*", "", addr, flags=re.I)
    addr = re.sub(r"\bC\/O\b.*", "", addr, flags=re.I)
    addr = re.sub(r"\b(near|opp|opposite|next to|behind)\b.*", "", addr, flags=re.I)
    addr = re.sub(r"\+?\d[\d\-\s]{7,}\d", "", addr)

    addr = addr.replace(";", ",")
    addr = re.sub(r"\s+", " ", addr).strip(" ,.-")
    return addr

def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def best_locality(row):
    for col in ["CITY", "LOCAL_MUNICIPALITY", "METROPOLITAN_MUNICIPALITY", "DISTRICT_MUNICIPALITY"]:
        if col in row.index:
            v = row.get(col)
            if pd.notna(v):
                s = clean_text(v)
                if s and s.lower() != "nan":
                    return s
    return ""

def build_query(row):
    name = clean_text(row.get("PRACTICE_NAME"))
    addr = clean_address(row.get("ADDRESS"))
    loc  = best_locality(row)
    prov = clean_text(row.get(PROVINCE_COL))

    parts = []
    if addr:
        parts.append(addr)
    if loc:
        parts.append(loc)
    if prov:
        parts.append(prov)
    parts.append("South Africa")

    q = ", ".join([p for p in parts if p])

    # If address is weak, add name
    if len(addr) < 8 and name:
        q = ", ".join([name, q])

    return q

def opencage_geocode(query):
    params = {
        "q": query,
        "key": OPENCAGE_API_KEY,
        "limit": 1,
        "no_annotations": 0,
        "language": "en",
        "countrycode": "za",
    }
    r = requests.get(OPENCAGE_URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    if data.get("total_results", 0) < 1:
        return None
    return data["results"][0]

def extract_fields(result):
    geometry = result.get("geometry", {}) or {}
    components = result.get("components", {}) or {}
    conf = result.get("confidence", None)

    return {
        "GEOCODE_LAT": geometry.get("lat"),
        "GEOCODE_LON": geometry.get("lng"),
        "GEOCODE_CONFIDENCE": conf,
        "GEOCODE_FORMATTED": result.get("formatted"),
        "GEOCODE_STATE": components.get("state"),
        "GEOCODE_CITY": components.get("city") or components.get("town") or components.get("village"),
        "GEOCODE_SUBURB": components.get("suburb"),
        "GEOCODE_ROAD": components.get("road"),
        "GEOCODE_HOUSENUMBER": components.get("house_number"),
    }

def province_mismatch(input_prov, geocoded_state):
    a = clean_text(input_prov).lower()
    b = clean_text(geocoded_state).lower()
    if not a or not b:
        return False
    return not (a in b or b in a)


# =========================
# RUN TEST
# =========================
if not OPENCAGE_API_KEY:
    raise ValueError(
        "No OpenCage API key found. Paste it into OPENCAGE_API_KEY or set env var OPENCAGE_API_KEY and restart the kernel."
    )

df = pd.read_csv(FILE_PATH)

# focus on missing coords if desired
df["COORDS"] = df["COORDS"].replace({np.nan: None})
to_geocode = df[df["COORDS"].isna()].copy()

test = to_geocode.sample(min(N_TEST, len(to_geocode)), random_state=42).copy().reset_index(drop=True)

rows = []
for i, row in enumerate(test.to_dict("records"), start=1):
    q = build_query(pd.Series(row))
    out = {"GEOCODE_QUERY": q}

    try:
        res = opencage_geocode(q)
        if res is None:
            out.update({"GEOCODE_STATUS": "no_match"})
        else:
            fields = extract_fields(res)
            out.update(fields)
            out["GEOCODE_STATUS"] = "ok"

            # review flag
            conf = fields.get("GEOCODE_CONFIDENCE")
            prov_in = row.get(PROVINCE_COL, "")
            prov_out = fields.get("GEOCODE_STATE", "")

            review = False
            if conf is None or float(conf) < CONFIDENCE_REVIEW_THRESHOLD:
                review = True
            if province_mismatch(prov_in, prov_out):
                review = True
            if fields.get("GEOCODE_LAT") is None or fields.get("GEOCODE_LON") is None:
                review = True

            out["GEOCODE_REVIEW_FLAG"] = review

    except Exception as e:
        out.update({"GEOCODE_STATUS": f"error: {type(e).__name__}", "GEOCODE_REVIEW_FLAG": True})

    rows.append(out)

    time.sleep(SLEEP_SECONDS)
    if i % 10 == 0:
        print(f"Done {i}/{len(test)}")

geo_df = pd.DataFrame(rows)
test_out = pd.concat([test, geo_df], axis=1)

print("\n=== SUMMARY ===")
print("Total:", len(test_out))
print("Matches:", test_out["GEOCODE_LAT"].notna().sum())
print("No match:", (test_out["GEOCODE_STATUS"] == "no_match").sum())
print("Errors:", test_out["GEOCODE_STATUS"].astype(str).str.startswith("error:").sum())
print("Review flagged:", (test_out.get("GEOCODE_REVIEW_FLAG", False) == True).sum())

# Show a preview
cols_show = [
    "PRACTICE_NAME", "ADDRESS", "CITY", "PROVINCE",
    "GEOCODE_STATUS", "GEOCODE_CONFIDENCE", "GEOCODE_STATE",
    "GEOCODE_LAT", "GEOCODE_LON", "GEOCODE_FORMATTED", "GEOCODE_QUERY"
]
cols_show = [c for c in cols_show if c in test_out.columns]
test_out[cols_show].head(20)

Done 10/100
Done 20/100
Done 30/100
Done 40/100
Done 50/100
Done 60/100
Done 70/100
Done 80/100
Done 90/100
Done 100/100

=== SUMMARY ===
Total: 100
Matches: 83
No match: 17
Errors: 0
Review flagged: 13


,PRACTICE_NAME,ADDRESS,CITY,PROVINCE,GEOCODE_STATUS,GEOCODE_CONFIDENCE,GEOCODE_STATE,GEOCODE_LAT,GEOCODE_LON,GEOCODE_FORMATTED,GEOCODE_QUERY
0,Eupham Pharmacy,"Shop 7 Falala Shopping Complex, Phumulani Cres...",NaN,Gauteng,no_match,NaN,NaN,NaN,NaN,NaN,"Shop 7 Falala Shopping Complex, Phumulani Cres..."
1,Eastern Medicine Depot,"24 Tom Jones Street, Benoni, Benoni",NaN,Gauteng,ok,9.0,Gauteng,-26.189972,28.316727,"Tom Jones Street, Kleinfontein Lake, Benoni, 1...","24 Tom Jones Street, Benoni, Benoni, Gauteng, ..."
2,Md24 Pharmacy,"Shop 3, Shelly Boulevard, Shelly Beach, Shelly...",NaN,KwaZulu-Natal,ok,9.0,KwaZulu-Natal,-30.807918,30.410757,"Shelly Boulevard, East Street, Hibiscus Coast ...","Shop 3, Shelly Boulevard, Shelly Beach, Shelly..."
3,Dis-Chem Bredell Square Pharmacy,"Shop 2, Bredell Square, Bredell, Kempton Park",NaN,Gauteng,ok,7.0,Gauteng,-26.108590,28.237700,"Kempton Park, Ekurhuleni Metropolitan Municipa...","Shop 2, Bredell Square, Bredell, Kempton Park,..."
4,Vistpharm Pharmacy,"Vista Clinic, Lyttelton, Centurion",NaN,Gauteng,ok,7.0,Gauteng,-25.858910,28.185770,"Centurion, City of Tshwane Metropolitan Munici...","Vista Clinic, Lyttelton, Centurion, Gauteng, S..."
5,St Johns Main Pharmacy,"Shops 10-11 Checkers Centre, 100 St Johns Aven...",NaN,KwaZulu-Natal,ok,7.0,KwaZulu-Natal,-29.820680,30.886740,"Pinetown, eThekwini Metropolitan Municipality,...","Shops 10-11 Checkers Centre, 100 St Johns Aven..."
6,Clicks Pharmacy - Carlton Centre,"Shop 123 Carlton Centre, 150 Commissioner Stre...",NaN,Gauteng,ok,1.0,Gauteng,-26.202270,28.043630,"Johannesburg, City of Johannesburg Metropolita...","Shop 123 Carlton Centre, 150 Commissioner Stre..."
7,Dis-Chem Cosmo Mall Pharmacy,"Shop No 152 Cosmo Mall, Corner Malibongwe Driv...",NaN,Gauteng,no_match,NaN,NaN,NaN,NaN,NaN,"Shop No 152 Cosmo Mall, Corner Malibongwe Driv..."
8,Top Level Pharmacy,"Shop 5 Top Level Centre, Fitzsimons Street, Va...",NaN,Gauteng,ok,6.0,Gauteng,-26.711710,27.837950,"Vanderbijlpark, Sedibeng District Municipality...","Shop 5 Top Level Centre, Fitzsimons Street, Va..."
9,Paruks Pharmacy,"Shop 1 Kenilworth Park, 202 Felix Dlamini Road...",NaN,KwaZulu-Natal,ok,7.0,KwaZulu-Natal,-29.857900,31.029200,"Durban, eThekwini Metropolitan Municipality, S...","Shop 1 Kenilworth Park, 202 Felix Dlamini Road..."
